In [0]:
-- Game - Penalty Summary
-- Every penalty in one game: when, who took it, the infraction, minutes and who drew it.
-- Change the values in params only; everything below reads from it.
WITH params AS (                              -- change settings here only
  SELECT
    2026020278 AS game_id   -- e.g. 2025020001 (season 2025, 02 = regular season, game 0001); see Game - Find Game ID
),
game_players AS (                        -- everyone dressed in the game (game rosters)
  SELECT r.game_id, r.player_id, concat_ws(' ', r.first_name, r.last_name) AS full_name,
         r.position_code, r.sweater_number, r.team_id, t.abbrev AS team,
         r.team_id = g.home_team_id AS is_home, g.home_team_id
  FROM nhl.silver.game_rosters r
  JOIN nhl.gold.dim_game g ON r.game_id = g.game_id
  JOIN nhl.gold.dim_team t ON r.team_id = t.team_id
  CROSS JOIN params pr
  WHERE r.game_id = pr.game_id
),
plays AS (                               -- every play in the game, with the owning team's strength
  SELECT t.abbrev AS team,
         f.event_owner_team_id = g.home_team_id AS owner_is_home,
         IF(f.event_owner_team_id = g.home_team_id, f.situation_code, reverse(f.situation_code)) AS owner_situation,
         CASE WHEN substr(owner_situation, 1, 1) = '0' THEN 'EN'      -- opponent net empty
              WHEN substr(owner_situation, 3, 1) > substr(owner_situation, 2, 1)
               AND substr(owner_situation, 4, 1) = '1' THEN 'PP'      -- more skaters, own goalie in
              WHEN substr(owner_situation, 3, 1) < substr(owner_situation, 2, 1) THEN 'SH'
              ELSE 'EV' END AS strength,
         CASE WHEN f.period_type IN ('OT', 'SO') THEN f.period_type
              ELSE CAST(f.period_number AS STRING) END AS period,
         SQRT(POW(89 - ABS(f.x_coord), 2) + POW(f.y_coord, 2)) AS distance_ft,
         f.*
  FROM nhl.gold.fact_event f
  JOIN nhl.gold.dim_game g ON f.game_id = g.game_id
  LEFT JOIN nhl.gold.dim_team t ON t.team_id = f.event_owner_team_id
  CROSS JOIN params pr
  WHERE f.game_id = pr.game_id
)
SELECT p.period,
       p.time_in_period          AS time,
       p.team,
       c.full_name               AS committed_by,
       p.penalty_desc_key        AS penalty,
       p.penalty_type_code       AS penalty_type,
       p.penalty_duration        AS minutes,
       d.full_name               AS drawn_by,
       sv.full_name              AS served_by
FROM plays p
LEFT JOIN game_players c  ON c.player_id  = p.committed_by_player_id
LEFT JOIN game_players d  ON d.player_id  = p.drawn_by_player_id
LEFT JOIN game_players sv ON sv.player_id = p.served_by_player_id
WHERE p.type_desc_key = 'penalty'
ORDER BY p.period_number, p.sort_order;